# Use case 6: who may *notify* on Aviso 2 for DestinE, and a playground for the DestinE Annual meeting

Two questions from the DestinE use-case list, answered with live probes and generated artefacts:

* **A. Which DESP users should be enabled to notify other listeners or software?** The server
  decides this per stream with realm-scoped roles. Part A explains the model, probes what the
  tokens on this machine can do today on the LUMI server and on the ECMWF playground, and turns
  the result into a draft role proposal (`uc06/stream-auth-proposal.yaml`).
* **B. Set up a playground Aviso server for the DestinE Annual meeting.** Part B produces the
  configuration for a self-contained demo deployment (schema, auth-o-tron realm, server config,
  a seeding script, a participant quick-start) and rehearses the seeding against the ECMWF
  playground.

| Setting | Value |
| --- | --- |
| LUMI Aviso 2 server | `https://aviso2.lumi.apps.dte.destination-earth.eu` (event type `data`) |
| ECMWF playground | `https://aviso-playground.ecmwf.int` (event types `mars`, `dissemination`, `de-scm-data`, `de-scm-data-points`, `extreme_event`, ...) |
| Tokens probed | DESP offline token (`~/.polytopeapirc`, `user_key`) and ECMWF API key (`~/.config/aviso/config.yaml`, `auth.bearer_token`) |
| Client | `pyaviso` 2.x (`pip install pyaviso pyyaml pandas`) |
| Sources | `aviso-server/docs/src/authentication.md`, `configuration-reference.md`, `schema-guide.md`, `aviso-deployment/aviso-chart/values_store/lumi-databridge/aviso-dev/values.yaml` |

Nothing valid is ever published to the LUMI server from this notebook. Test notifications go to
the shared playground only, are clearly marked, and are behind opt-in flags. Tokens are never
printed.

## Part A. The permission model

Authentication and authorisation in Aviso 2 (`aviso-server`) work like this:

1. The client sends `Authorization: Bearer <token>` (or Basic credentials).
2. In **direct** mode the server forwards the header to **auth-o-tron**, which knows several
   *providers*, one per **realm**, and returns a signed JWT carrying `username`, `realm` and
   `roles`. On the LUMI data bridge the providers are (Helm values of `aviso-dev`):

   | Provider | Type | Realm | Who authenticates |
   | --- | --- | --- | --- |
   | `destine` | `openid-offline` (public client `polytope-api-public`, IAM `https://auth.destine.eu`) | `desp` | every DESP account, with the same offline token Polytope uses |
   | `ECMWF API Provider` | `ecmwf-api` (`https://api.ecmwf.int/v1`) | `ecmwf` | ECMWF API key holders |
   | `plain-provider` | `plain` | `localrealm` | a couple of hard-coded operator accounts |

3. Route handlers then apply the **per-stream** rules from the schema's `auth:` block:

   | `auth.required` | `read_roles` | `write_roles` | Read (`watch`/`replay`) | Write (`notify`) |
   | --- | --- | --- | --- | --- |
   | `false` / omitted | any | any | anyone | anyone |
   | `true` | omitted | omitted | any authenticated user | **admins only** |
   | `true` | set | omitted | must match `read_roles` | **admins only** |
   | `true` | omitted | set | any authenticated user | must match `write_roles` or be admin |
   | `true` | set | set | must match `read_roles` | must match `write_roles` or be admin |

   Roles are realm-scoped maps (`write_roles: {desp: ["dt-producer"]}`); `["*"]` grants a whole
   realm. Global `auth.admin_roles` always grants both. Roles come from auth-o-tron
   *augmenters* (e.g. `plain_advanced` matching usernames), not from the schema.
4. `401 UNAUTHORIZED` = no or bad token; `403 FORBIDDEN` = valid token but the user lacks the
   role. Schema endpoints are always public.
5. The **ECPDS plugin** (`auth.plugins: ["ecpds"]`) additionally checks the requested
   `destination` against the user's ECPDS destination list, but only on **reads**. `notify` is
   never gated by ECPDS.

So "who may notify" is exactly the set `admin_roles ∪ write_roles` of each stream. Today the
LUMI `data` stream and the playground's DestinE streams have no `write_roles` visible to
clients (the `auth` block is not exposed by `GET /schema`), which is what the probes below test.

In [1]:
import json
import os
import pathlib
import time
from datetime import datetime, timezone

import pandas as pd
import pyaviso
import yaml
from IPython.display import display

print("pyaviso", pyaviso.__version__)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)


class NoAliasDumper(yaml.SafeDumper):
    # shared dicts (read_roles etc.) must be written out in full, not as YAML anchors
    def ignore_aliases(self, data):
        return True


def to_yaml(doc):
    return yaml.dump(doc, Dumper=NoAliasDumper, sort_keys=False)


SERVERS = {
    "lumi": "https://aviso2.lumi.apps.dte.destination-earth.eu",
    "playground": "https://aviso-playground.ecmwf.int",
}
OUT = pathlib.Path("uc06")
OUT.mkdir(exist_ok=True)


def token_candidates():
    # (label, token) pairs; secrets stay in memory only and are never printed.
    out = []
    if os.environ.get("AVISO_TOKEN"):
        out.append(("AVISO_TOKEN env", os.environ["AVISO_TOKEN"]))
    rc = pathlib.Path.home() / ".polytopeapirc"
    if rc.exists():
        key = json.loads(rc.read_text(encoding="utf-8")).get("user_key")
        if key:
            out.append(("DESP offline token", key))
    cfg = pathlib.Path.home() / ".config" / "aviso" / "config.yaml"
    if cfg.exists():
        key = ((yaml.safe_load(cfg.read_text(encoding="utf-8")) or {}).get("auth") or {}).get("bearer_token")
        if key:
            out.append(("ECMWF API key", key))
    return out


TOKENS = token_candidates()
print("tokens available:", [(label, f"{len(tok)} chars") for label, tok in TOKENS])


def open_stream(client, event_type, *, filter, start=None, mode=None):
    # pyaviso 2.1.0 names the cursor `start_from=`; newer builds `from_=`.
    kwargs = {"filter": filter}
    if mode is not None:
        kwargs["mode"] = mode
    if start is None:
        return client.listen(event_type, **kwargs)
    try:
        return client.listen(event_type, start_from=start, **kwargs)
    except TypeError:
        return client.listen(event_type, from_=start, **kwargs)

pyaviso 2.1.0
tokens available: [('DESP offline token', '605 chars'), ('ECMWF API key', '32 chars')]


### Live probe matrix

For every (server, token) pair three operations are tried:

* **schema**: `GET /api/v1/schema`, anonymous, expected to work everywhere.
* **replay**: a one-notification replay (read permission).
* **notify** (write permission):
  * on **LUMI** the request is *deliberately invalid* (`data` event without the mandatory
    `class`). The server never stores anything. `401`/`403` means the token may not write;
    `400 INVALID_NOTIFICATION_REQUEST` means the request passed authentication and
    authorisation and failed only validation, so the token **could** write. The auth middleware
    runs before validation (`authentication.md`, "How it works"), so a 400 is a conclusive
    "write allowed"; still, treat it as *inferred*, not observed.
  * on the **playground** (a shared sandbox) a clearly marked test notification is published on
    `de-scm-data` (`experiment=uc6test`). This is the observed write permission.

Statuses are recorded, never the tokens.

In [2]:
PROBE_EVENT = {"lumi": "data", "playground": "de-scm-data"}
PROBE_FILTER = {"lumi": {"class": "d1"}, "playground": {"experiment": "*"}}
now = datetime.now(timezone.utc)
PLAYGROUND_TEST_ID = {"experiment": "uc6test", "date": now.strftime("%Y%m%d"), "time": now.strftime("%H%M")}
PLAYGROUND_TEST_PAYLOAD = {"output_path": "/tmp/aviso-uc6-notebook-test", "note": "permission probe from uc06 notebook"}


def status_of(exc):
    if isinstance(exc, pyaviso.HttpError):
        try:
            body = json.loads(exc.body) if exc.body else {}
        except json.JSONDecodeError:
            body = {}
        return f"HTTP {exc.status} {body.get('code', '')}".strip()
    return f"{type(exc).__name__}"


def probe_schema(base):
    try:
        cat = pyaviso.AvisoClient(base_url=base).schema()
        return "200 " + ",".join(sorted(cat.event_types))
    except pyaviso.AvisoError as exc:
        return status_of(exc)


def probe_replay(base, server, tok):
    client = pyaviso.AvisoClient(base_url=base, auth=pyaviso.Bearer(tok) if tok else None)
    try:
        with open_stream(client, PROBE_EVENT[server], filter=PROBE_FILTER[server], start=1, mode="replay_only") as s:
            n = next(iter(s), None)
        return "200 read OK" + ("" if n else " (stream empty)")
    except pyaviso.AvisoError as exc:
        return status_of(exc)


def probe_notify(base, server, tok):
    client = pyaviso.AvisoClient(base_url=base, auth=pyaviso.Bearer(tok) if tok else None)
    try:
        if server == "lumi":
            # invalid on purpose: `class` is missing, so nothing can be stored
            client.notify(event_type="data", identifier={"date": now.strftime("%Y%m%d")}, payload=None)
            return "UNEXPECTED 2xx (invalid body accepted!)"
        r = client.notify(event_type="de-scm-data", identifier=PLAYGROUND_TEST_ID, payload=PLAYGROUND_TEST_PAYLOAD)
        return f"{r.status} write OK"
    except pyaviso.AvisoError as exc:
        s = status_of(exc)
        if server == "lumi" and s.startswith("HTTP 400"):
            s += " -> write allowed (inferred)"
        return s


rows = []
for server, base in SERVERS.items():
    rows.append({"server": server, "token": "(anonymous)", "schema": probe_schema(base),
                 "replay": probe_replay(base, server, None), "notify": probe_notify(base, server, None)})
    for label, tok in TOKENS:
        rows.append({"server": server, "token": label, "schema": "(same)",
                     "replay": probe_replay(base, server, tok), "notify": probe_notify(base, server, tok)})
matrix = pd.DataFrame(rows).set_index(["server", "token"])
pd.set_option("display.max_colwidth", 80)
display(matrix)
matrix.to_csv(OUT / "probe-matrix.csv")

schema                 replay  \
server     token                                                                                                                        
lumi       (anonymous)                                                                                200 data  HTTP 401 UNAUTHORIZED   
           DESP offline token                                                                           (same)            200 read OK   
           ECMWF API key                                                                                (same)            200 read OK   
playground (anonymous)         200 cleareo_app1_data_available,cleareo_app1_extreme_rainfall,de-scm-data,de...  HTTP 401 UNAUTHORIZED   
           DESP offline token                                                                           (same)              AuthError   
           ECMWF API key                                                                                (same)            200 read OK   

                                                                                          notify  
server     token                                                                                  
lumi       (anonymous)                                                     HTTP 401 UNAUTHORIZED  
           DESP offline token                                                 HTTP 403 FORBIDDEN  
           ECMWF API key       HTTP 400 INVALID_NOTIFICATION_REQUEST -> write allowed (inferred)  
playground (anonymous)                                                     HTTP 401 UNAUTHORIZED  
           DESP offline token                                              HTTP 401 UNAUTHORIZED  
           ECMWF API key                                                        success write OK

In [3]:
# Reading the matrix: which (server, token) pairs can read, and which can write.
readers = matrix[matrix["replay"].str.startswith("200")].index.tolist()
writers = matrix[matrix["notify"].str.contains("write OK|write allowed")].index.tolist()
print("can read :", readers)
print("can write:", writers)
print()
for (server, token), row in matrix.iterrows():
    if token == "(anonymous)":
        continue
    read = row["replay"].startswith("200")
    write = "write OK" in row["notify"] or "write allowed" in row["notify"]
    verdict = ("producer (read + write)" if read and write else
               "consumer only (read, no write role)" if read else
               "not accepted by this server")
    print(f"{server:<10} {token:<20} -> {verdict:<38} [{row['notify']}]")

can read : [('lumi', 'DESP offline token'), ('lumi', 'ECMWF API key'), ('playground', 'ECMWF API key')]
can write: [('lumi', 'ECMWF API key'), ('playground', 'ECMWF API key')]

lumi       DESP offline token   -> consumer only (read, no write role)    [HTTP 403 FORBIDDEN]
lumi       ECMWF API key        -> producer (read + write)                [HTTP 400 INVALID_NOTIFICATION_REQUEST -> write allowed (inferred)]
playground DESP offline token   -> not accepted by this server            [HTTP 401 UNAUTHORIZED]
playground ECMWF API key        -> producer (read + write)                [success write OK]


How to read the matrix (observed on 2026-09-24, re-run the cells for today's state):

* **LUMI, DESP offline token**: read OK, notify `403 FORBIDDEN`. A DESP user is authenticated
  (realm `desp`) but holds no write role on `data`. This is the *consumer* profile and is the
  right default for NMHSs, businesses and researchers.
* **LUMI, ECMWF API key**: read OK, notify `400 INVALID_NOTIFICATION_REQUEST` on the deliberately
  broken body, so the request passed authentication and authorisation: an ECMWF realm user
  currently **may publish** on `data`. Whether this is intended (staff as producers) or too
  broad is one of the decisions for the team.
* **Playground, DESP token**: rejected (`401` / client `AuthError`): the playground is wired to
  the ECMWF realm only; a DESP realm would have to be added to its auth-o-tron before the
  Annual meeting participants can log in with their DESP accounts (Part B does that).
* **Playground, ECMWF key**: full producer rights on `de-scm-data` (the test notification was
  stored; `experiment=uc6test`).

### Draft recommendation: who should be able to notify

Based on the model above and on the DestinE actors in the use-case list, a role proposal per
stream. Realm `desp` is every DESP account; roles are assigned in auth-o-tron (augmenters or the
DESP IAM groups), not by users themselves. **Consumers never need write access**: NMHSs,
businesses and researchers only `watch`/`replay`.

| Stream (event type) | Producers who should hold a `write` role | Readers |
| --- | --- | --- |
| `data` (DT output landed on a data bridge) | the data-bridge transfer service (ECPDS or the DT ingestion pipeline), realm `ecmwf` service account; DT operators (`desp: dt-operator`) | all authenticated DESP users (`desp: ["*"]`) |
| `dissemination` (ECPDS products) | ECPDS service account only (`ecmwf: ecpds-producer`) | ECPDS destination owners; enforce with `plugins: ["ecpds"]` |
| `de-scm-data`, `de-scm-data-points` (experiment output, spatial) | experiment owners / SCM operators (`desp: scm-operator`, `ecmwf: scm-operator`) | all authenticated users |
| `extreme_event` / regional alerts (DEODE, DE330) | DEODE on-demand run operators at NMHSs (`desp: deode-producer`) | all authenticated users, filtered spatially |
| admin endpoints | Aviso operators only (`admin_roles`) | n/a |

Principles: one **service account per producing system**, roles named after the system
(`*-producer`, `*-operator`), no wildcard write roles, `auth.required: true` on every DestinE
stream, and a **staging stream** (`*-test`) with a wildcard write role for onboarding new
producers without touching the operational ones. The cell below turns this into the `auth:`
blocks the server expects.

In [4]:
PROPOSAL = {
    "data": {
        "required": True,
        "read_roles": {"desp": ["*"], "ecmwf": ["*"]},
        "write_roles": {"ecmwf": ["ecpds-producer", "dt-ingest"], "desp": ["dt-operator"]},
    },
    "dissemination": {
        "required": True,
        "read_roles": {"ecmwf": ["*"], "desp": ["*"]},
        "write_roles": {"ecmwf": ["ecpds-producer"]},
        "plugins": ["ecpds"],
    },
    "de-scm-data": {
        "required": True,
        "read_roles": {"desp": ["*"], "ecmwf": ["*"]},
        "write_roles": {"desp": ["scm-operator"], "ecmwf": ["scm-operator"]},
    },
    "de-scm-data-points": {
        "required": True,
        "read_roles": {"desp": ["*"], "ecmwf": ["*"]},
        "write_roles": {"desp": ["scm-operator"], "ecmwf": ["scm-operator"]},
    },
    "extreme_event": {
        "required": True,
        "read_roles": {"desp": ["*"], "ecmwf": ["*"]},
        "write_roles": {"desp": ["deode-producer"], "ecmwf": ["deode-producer"]},
    },
    "destine-test": {
        "required": True,
        "read_roles": {"desp": ["*"], "ecmwf": ["*"]},
        "write_roles": {"desp": ["*"], "ecmwf": ["*"]},   # onboarding sandbox: every authenticated user may publish
    },
}
GLOBAL_AUTH = {"enabled": True, "mode": "direct", "auth_o_tron_url": "http://auth-o-tron-svc:8080",
               "jwt_secret": "<shared with auth-o-tron jwt.secret>", "admin_roles": {"ecmwf": ["aviso-admin"], "desp": ["aviso-admin"]}}

proposal_doc = {"auth": GLOBAL_AUTH, "notification_schema": {ev: {"auth": block} for ev, block in PROPOSAL.items()}}
text = to_yaml(proposal_doc)
(OUT / "stream-auth-proposal.yaml").write_text(text, encoding="utf-8")
assert yaml.safe_load(text)["notification_schema"]["dissemination"]["auth"]["plugins"] == ["ecpds"]
print(text)

auth:
  enabled: true
  mode: direct
  auth_o_tron_url: http://auth-o-tron-svc:8080
  jwt_secret: <shared with auth-o-tron jwt.secret>
  admin_roles:
    ecmwf:
    - aviso-admin
    desp:
    - aviso-admin
notification_schema:
  data:
    auth:
      required: true
      read_roles:
        desp:
        - '*'
        ecmwf:
        - '*'
      write_roles:
        ecmwf:
        - ecpds-producer
        - dt-ingest
        desp:
        - dt-operator
  dissemination:
    auth:
      required: true
      read_roles:
        ecmwf:
        - '*'
        desp:
        - '*'
      write_roles:
        ecmwf:
        - ecpds-producer
      plugins:
      - ecpds
  de-scm-data:
    auth:
      required: true
      read_roles:
        desp:
        - '*'
        ecmwf:
        - '*'
      write_roles:
        desp:
        - scm-operator
        ecmwf:
        - scm-operator
  de-scm-data-points:
    auth:
      required: true
      read_roles:
        desp:
        - '*'
        ecmwf:
   

## Part B. A playground Aviso server for the DestinE Annual meeting

Goal: a server participants can publish to and listen on during the meeting, with DestinE-shaped
streams, DESP logins, and pre-seeded history, without touching the operational LUMI server.

**Checklist**

1. **Where**: a Kubernetes namespace with the Helm chart (`aviso-deployment/aviso-chart`,
   `values_store/lumi-databridge/aviso-dev` is the closest template) or, for a laptop-sized
   demo, `aviso-server` + NATS JetStream via `scripts/init_nats.sh` and auth-o-tron via
   `scripts/auth-o-tron-docker.sh`. JetStream is required for replay history that survives
   restarts (`deployment-modes.md`).
2. **Schema**: the DestinE streams below (`uc06/meeting-schema.yaml`). Each stream has
   `auth.required: true`; `destine-demo` lets every authenticated participant publish, the
   others are read-only for participants and writable by the demo operators.
3. **Auth**: auth-o-tron with the DESP `openid-offline` provider (realm `desp`, public client
   `polytope-api-public`) so participants use their DESP offline token, plus the ECMWF provider
   for staff and a `plain` realm for the demo operator (`uc06/meeting-auth-o-tron.yaml`).
4. **Server config**: JetStream backend, 10-day retention, base URL, admin roles
   (`uc06/meeting-aviso-config.yaml`).
5. **Seed history** the evening before with `uc06/seed_demo_events.py` so `replay` shows
   something immediately; rehearsed below against the ECMWF playground.
6. **Participant quick-start** (last section): token, CLI config, three commands.
7. **After the meeting**: `aviso` admin `wipe/all` or delete the namespace; revoke the demo
   operator account.

In [5]:
MARS_ID = {
    "class": {"type": "StringHandler", "max_length": 2, "required": True},
    "date": {"type": "DateHandler", "required": False},
    "time": {"type": "TimeHandler", "required": False},
    "expver": {"type": "ExpverHandler", "required": False},
    "domain": {"type": "EnumHandler", "values": ["g"], "required": False},
    "stream": {"type": "EnumHandler", "values": ["oper", "wave"], "required": False},
    "type": {"type": "StringHandler", "max_length": 2, "required": False},
    "levtype": {"type": "EnumHandler", "values": ["sfc", "pl", "hl"], "required": False},
    "step": {"type": "IntHandler", "range": [0, 10000], "required": False},
}
DESP_READ = {"desp": ["*"], "ecmwf": ["*"], "localrealm": ["*"]}
OPERATOR_WRITE = {"ecmwf": ["demo-operator"], "localrealm": ["demo-operator"]}
RETENTION = {"retention_time": "10d", "max_messages": 200000, "max_size": "1Gi", "allow_duplicates": True}

MEETING_SCHEMA = {
    "notification_schema": {
        "data": {  # mirrors the LUMI data-bridge stream
            "topic": {"base": "de-data", "key_order": ["class", "expver", "stream", "type", "levtype", "date", "time", "step"]},
            "identifier": MARS_ID,
            "payload": {"required": False},
            "auth": {"required": True, "read_roles": DESP_READ, "write_roles": OPERATOR_WRITE},
            "storage_policy": RETENTION,
        },
        "de-scm-data": {
            "topic": {"base": "de-scm", "key_order": ["experiment", "date", "time"]},
            "identifier": {
                "experiment": {"type": "StringHandler", "max_length": 10, "required": True},
                "date": {"type": "DateHandler", "required": False},
                "time": {"type": "TimeHandler", "required": False},
            },
            "payload": {"required": True},
            "auth": {"required": True, "read_roles": DESP_READ, "write_roles": OPERATOR_WRITE},
            "storage_policy": RETENTION,
        },
        "de-scm-data-points": {
            "topic": {"base": "de-scm-points", "key_order": ["experiment", "date", "time"]},
            "identifier": {
                "experiment": {"type": "StringHandler", "max_length": 10, "required": True},
                "date": {"type": "DateHandler", "required": False},
                "time": {"type": "TimeHandler", "required": False},
                "point_cloud": {"type": "PointCloudHandler", "max_points": 10000, "required": True,
                                "description": "Publishers send [[lat, lon], ...]; subscribers send a closed polygon instead."},
            },
            "payload": {"required": False},
            "auth": {"required": True, "read_roles": DESP_READ, "write_roles": OPERATOR_WRITE},
            "storage_policy": RETENTION,
        },
        "extreme_event": {
            "topic": {"base": "de-extreme", "key_order": ["region", "run_time", "severity"]},
            "identifier": {
                "region": {"type": "EnumHandler", "values": ["north", "south", "east", "west"], "required": True},
                "run_time": {"type": "TimeHandler", "required": True},
                "severity": {"type": "IntHandler", "range": [1, 7], "required": True},
                "anomaly": {"type": "FloatHandler", "range": [0.0, 100.0], "required": False},
                "polygon": {"type": "PolygonHandler", "required": True},
            },
            "payload": {"required": False},
            "auth": {"required": True, "read_roles": DESP_READ, "write_roles": OPERATOR_WRITE},
            "storage_policy": RETENTION,
        },
        "destine-demo": {  # every participant may publish here
            "topic": {"base": "de-demo", "key_order": ["team", "date", "time"]},
            "identifier": {
                "team": {"type": "StringHandler", "max_length": 20, "required": True},
                "date": {"type": "DateHandler", "required": False},
                "time": {"type": "TimeHandler", "required": False},
            },
            "payload": {"required": True},
            "auth": {"required": True, "read_roles": DESP_READ, "write_roles": {"desp": ["*"], "ecmwf": ["*"], "localrealm": ["*"]}},
            "storage_policy": {"retention_time": "2d", "max_messages": 50000, "max_size": "256Mi", "allow_duplicates": True},
        },
    }
}
schema_text = to_yaml(MEETING_SCHEMA)
assert "&id" not in schema_text
(OUT / "meeting-schema.yaml").write_text(schema_text, encoding="utf-8")
loaded = yaml.safe_load(schema_text)["notification_schema"]
bases = [v["topic"]["base"] for v in loaded.values()]
assert len(bases) == len(set(bases)), "topic bases must be unique"
for ev, spec in loaded.items():
    assert spec["auth"]["required"] is True and set(spec["auth"]["write_roles"]), ev
print(f"{len(loaded)} streams -> {OUT / 'meeting-schema.yaml'} ({len(schema_text)} bytes); topic bases: {bases}")
print(schema_text[:1200], "...")

5 streams -> uc06/meeting-schema.yaml (4845 bytes); topic bases: ['de-data', 'de-scm', 'de-scm-points', 'de-extreme', 'de-demo']
notification_schema:
  data:
    topic:
      base: de-data
      key_order:
      - class
      - expver
      - stream
      - type
      - levtype
      - date
      - time
      - step
    identifier:
      class:
        type: StringHandler
        max_length: 2
        required: true
      date:
        type: DateHandler
        required: false
      time:
        type: TimeHandler
        required: false
      expver:
        type: ExpverHandler
        required: false
      domain:
        type: EnumHandler
        values:
        - g
        required: false
      stream:
        type: EnumHandler
        values:
        - oper
        - wave
        required: false
      type:
        type: StringHandler
        max_length: 2
        required: false
      levtype:
        type: EnumHandler
        values:
        - sfc
        - pl
        - hl
     

In [6]:
AUTH_O_TRON = {
    "version": "1.0.0",
    "logging": {"level": "info", "format": "console"},
    "auth": {"timeout_in_ms": 3000},
    "providers": [
        {"name": "destine", "type": "openid-offline", "realm": "desp",
         "public_client_id": "polytope-api-public", "private_client_id": "polytope-api-bearer",
         "private_client_secret": "${DESP_PRIVATE_CLIENT_SECRET}",
         "cert_uri": "https://auth.destine.eu/realms/desp/protocol/openid-connect/certs",
         "iam_url": "https://auth.destine.eu"},
        {"name": "ecmwf-api", "type": "ecmwf-api", "realm": "ecmwf", "uri": "https://api.ecmwf.int/v1"},
        {"name": "plain-provider", "type": "plain", "realm": "localrealm",
         "users": [{"username": "demo-operator", "password": "${DEMO_OPERATOR_PASSWORD}"}]},
    ],
    "augmenters": [
        {"name": "ecmwf-roles", "type": "plain_advanced", "realm": "ecmwf",
         "match": {"username": ["<ecmwf-usernames-of-the-demo-team>"]}, "augment": {"roles": ["demo-operator", "aviso-admin"]}},
        {"name": "local-roles", "type": "plain", "realm": "localrealm",
         "roles": {"demo-operator": ["demo-operator"], "aviso-admin": ["demo-operator"]}},
    ],
    "store": {"enabled": False},
    "services": [],
    "jwt": {"exp": 3600, "iss": "authotron-destine-meeting", "aud": "authotron-destine-meeting", "secret": "${JWT_SECRET}"},
    "bind_address": "0.0.0.0:8080",
}
AVISO_CONFIG = {
    "application": {"host": "0.0.0.0", "port": 8000, "base_url": "https://aviso-meeting.example.destination-earth.eu"},
    "watch_endpoint": {"sse_heartbeat_interval_sec": 30, "connection_max_duration_sec": 3600,
                       "replay_batch_size": 100, "max_historical_notifications": 10000},
    "notification_backend": {"kind": "jetstream", "jetstream": {"nats_url": "nats://nats:4222", "token": "${NATS_TOKEN}"}},
    "auth": {"enabled": True, "mode": "direct", "auth_o_tron_url": "http://auth-o-tron:8080",
             "jwt_secret": "${JWT_SECRET}", "admin_roles": {"ecmwf": ["aviso-admin"], "localrealm": ["aviso-admin"]}, "timeout_ms": 5000},
    "logging": {"level": "info", "format": "json"},
}
AVISO_CONFIG.update(MEETING_SCHEMA)
for name, doc in [("meeting-auth-o-tron.yaml", AUTH_O_TRON), ("meeting-aviso-config.yaml", AVISO_CONFIG)]:
    t = to_yaml(doc)
    (OUT / name).write_text(t, encoding="utf-8")
    assert yaml.safe_load(t)
    print(f"wrote {OUT / name} ({len(t)} bytes)")
print(to_yaml({"auth": AVISO_CONFIG["auth"]}))

wrote uc06/meeting-auth-o-tron.yaml (1118 bytes)
wrote uc06/meeting-aviso-config.yaml (5450 bytes)
auth:
  enabled: true
  mode: direct
  auth_o_tron_url: http://auth-o-tron:8080
  jwt_secret: ${JWT_SECRET}
  admin_roles:
    ecmwf:
    - aviso-admin
    localrealm:
    - aviso-admin
  timeout_ms: 5000



Bring it up (laptop-sized rehearsal; the Helm chart does the same on Kubernetes):

```bash
# 1. NATS JetStream with a token (prints NATS_TOKEN)
ENABLE_AUTH=true ./aviso-server/scripts/init_nats.sh
# 2. auth-o-tron with the meeting realm file (fill the ${...} secrets first)
AUTH_O_TRON_CONFIG_FILE=$PWD/uc06/meeting-auth-o-tron.yaml ./aviso-server/scripts/auth-o-tron-docker.sh start --detach
# 3. the server
docker run --rm -p 8000:8000 -v $PWD/uc06/meeting-aviso-config.yaml:/app/configuration/config.yaml:ro aviso-server:local
# 4. seed history (see next cell), then hand out the quick-start
```

### Seeding demo history

`uc06/seed_demo_events.py` publishes a small, clearly labelled batch with `notify_many`:
a few `de-scm-data` "experiment output landed" events and, when the target server has the
stream, `destine-demo` events. It takes the server from `AVISO_BASE_URL` and the token from
`AVISO_TOKEN` or the aviso CLI config. Below it is rehearsed against the **ECMWF playground**
(the meeting server does not exist yet), behind an opt-in flag, so the replay afterwards shows
what participants would see.

In [7]:
SEED_SCRIPT = r'''
# Seed a DestinE demo Aviso server with clearly labelled example notifications.
#
# Usage:
#     AVISO_BASE_URL=https://<server> AVISO_TOKEN=<token> python seed_demo_events.py [--tag TAG] [--count N]
#
# The token may also come from ~/.config/aviso/config.yaml (auth.bearer_token) or ~/.polytopeapirc (user_key).
import argparse
import json
import os
import pathlib
import sys
from datetime import datetime, timedelta, timezone

import pyaviso


def resolve_token():
    if os.environ.get("AVISO_TOKEN"):
        return os.environ["AVISO_TOKEN"]
    cfg = pathlib.Path.home() / ".config" / "aviso" / "config.yaml"
    if cfg.exists():
        import yaml
        tok = ((yaml.safe_load(cfg.read_text()) or {}).get("auth") or {}).get("bearer_token")
        if tok:
            return tok
    rc = pathlib.Path.home() / ".polytopeapirc"
    if rc.exists():
        return json.loads(rc.read_text()).get("user_key")
    sys.exit("no token: set AVISO_TOKEN")


def build_batch(event_types, tag, count):
    now = datetime.now(timezone.utc)
    batch = []
    for i in range(count):
        t = now - timedelta(hours=6 * i)
        if "de-scm-data" in event_types:
            batch.append({"event_type": "de-scm-data",
                          "identifier": {"experiment": tag[:10], "date": t.strftime("%Y%m%d"), "time": t.strftime("%H%M")},
                          "payload": {"output_path": f"ec:/RDX/prepIFS/{tag}/{t:%Y%m%d%H%M}/",
                                      "note": "DestinE Annual meeting demo data (seed_demo_events.py)"}})
        if "destine-demo" in event_types:
            batch.append({"event_type": "destine-demo",
                          "identifier": {"team": tag[:20], "date": t.strftime("%Y%m%d"), "time": t.strftime("%H%M")},
                          "payload": {"message": f"hello from {tag}", "note": "DestinE Annual meeting demo"}})
    return batch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tag", default="despdemo")
    ap.add_argument("--count", type=int, default=3)
    args = ap.parse_args()
    base = os.environ.get("AVISO_BASE_URL", "http://localhost:8000")
    client = pyaviso.AvisoClient(base_url=base, auth=pyaviso.Bearer(resolve_token()))
    available = set(client.schema().event_types)
    batch = build_batch(available, args.tag, args.count)
    if not batch:
        sys.exit(f"{base} has none of the demo streams (found {sorted(available)})")
    results = client.notify_many(batch)
    ok = sum(1 for r in results if r.ok)
    for r in results:
        item = batch[r.index]
        status = r.response.status if r.ok else f"FAILED {r.error}"
        print(f"[{r.index}] {item['event_type']} {item['identifier']} -> {status}")
    print(f"{ok}/{len(batch)} published to {base}")
    return 0 if ok == len(batch) else 1


if __name__ == "__main__":
    sys.exit(main())
'''
(OUT / "seed_demo_events.py").write_text(SEED_SCRIPT.lstrip("\n"), encoding="utf-8")
compile(SEED_SCRIPT, "seed_demo_events.py", "exec")   # syntax check without writing bytecode
print("wrote", OUT / "seed_demo_events.py")

wrote uc06/seed_demo_events.py


In [8]:
import subprocess
import sys

SEED_PLAYGROUND = True     # opt-in: publishes 3 labelled test notifications to the shared ECMWF playground
ecmwf_key = dict(TOKENS).get("ECMWF API key") or dict(TOKENS).get("AVISO_TOKEN env")
if SEED_PLAYGROUND and ecmwf_key:
    env = {**os.environ, "AVISO_BASE_URL": SERVERS["playground"], "AVISO_TOKEN": ecmwf_key}
    run = subprocess.run([sys.executable, str(OUT / "seed_demo_events.py"), "--tag", "despdemo", "--count", "3"],
                         env=env, capture_output=True, text=True, timeout=120)
    print(run.stdout, run.stderr[-800:], f"exit={run.returncode}")
else:
    print("seeding skipped (SEED_PLAYGROUND is False or no ECMWF key available)")

[0] de-scm-data {'experiment': 'despdemo', 'date': '20260924', 'time': '1538'} -> success
[1] de-scm-data {'experiment': 'despdemo', 'date': '20260924', 'time': '0938'} -> success
[2] de-scm-data {'experiment': 'despdemo', 'date': '20260924', 'time': '0338'} -> success
3/3 published to https://aviso-playground.ecmwf.int
  exit=0


In [9]:
# What a participant sees afterwards: replay the demo experiment on the playground.
if ecmwf_key:
    pg = pyaviso.AvisoClient(base_url=SERVERS["playground"], auth=pyaviso.Bearer(ecmwf_key))
    with open_stream(pg, "de-scm-data", filter={"experiment": "despdemo"}, start=1, mode="replay_only") as s:
        seeded = list(s)
    print(f"{len(seeded)} 'despdemo' notifications retained on the playground")
    display(pd.DataFrame([{"sequence": n.sequence, **n.identifier, "output_path": (n.payload or {}).get("output_path"),
                           "published": (n.cloudevent or {}).get("time")} for n in seeded]).tail(6))

6 'despdemo' notifications retained on the playground


,sequence,date,experiment,time,output_path,published
0,33,20260924,despdemo,0937,ec:/RDX/prepIFS/despdemo/202609240937/,2026-09-24T15:37:16.127275774Z
1,34,20260924,despdemo,0337,ec:/RDX/prepIFS/despdemo/202609240337/,2026-09-24T15:37:16.127325767Z
2,35,20260924,despdemo,1537,ec:/RDX/prepIFS/despdemo/202609241537/,2026-09-24T15:37:16.127335573Z
3,42,20260924,despdemo,1538,ec:/RDX/prepIFS/despdemo/202609241538/,2026-09-24T15:38:15.548727046Z
4,43,20260924,despdemo,0938,ec:/RDX/prepIFS/despdemo/202609240938/,2026-09-24T15:38:15.548776518Z
5,44,20260924,despdemo,0338,ec:/RDX/prepIFS/despdemo/202609240338/,2026-09-24T15:38:15.548784505Z


### Participant quick-start (hand-out)

```bash
pip install pyaviso                                   # client + `aviso` CLI
export AVISO_BASE_URL=https://aviso-meeting.example.destination-earth.eu
export AVISO_TOKEN="$(python -c 'import json,pathlib;print(json.load(open(pathlib.Path.home()/".polytopeapirc"))["user_key"])')"
#   ^ your DESP offline token (create it with desp-authentication.py); ECMWF staff use their API key instead

aviso schema list                                     # what streams exist (no token needed)
aviso replay --event de-scm-data --identifiers '{"experiment":"despdemo"}' --from 1 --json | cat
aviso listen --event destine-demo --identifiers '{"team":"*"}' --json     # live, Ctrl+C to stop
aviso notify 'event=destine-demo,team=<your-team>,date=20261001,time=1030,data={"message":"hello"}'
```

Python:

```python
import os, pyaviso
client = pyaviso.AvisoClient(base_url=os.environ["AVISO_BASE_URL"], auth=pyaviso.Env())
with client.listen("destine-demo", filter={"team": "*"}) as stream:
    for n in stream:
        print(n.sequence, n.identifier, n.payload)
```

### Open points for the team

* Assigning roles to DESP users needs an auth-o-tron **augmenter** rule per role (or DESP IAM
  group claims mapped to roles); decide who owns that list.
* The `auth:` blocks are not visible through `GET /schema`, so producers cannot discover their
  own permissions; a `403` on `notify` is the only signal. Consider documenting per-stream
  writers in the schema `description`.
* Playground retention on ECMWF's playground is 10 days / 100k messages per stream; the demo
  server config above uses the same numbers.

## Related Polytope examples

The DestinE Polytope examples repository (`~/Git/polytope/polytope-examples`, GitHub
`destination-earth-digital-twins/polytope-examples`) contains no Aviso code, but it is the
reference for the **data side** of every Aviso workflow: once a notification names a field, these
notebooks show how to fetch exactly that field (or a feature cut from it) with earthkit-data.

| Need | Example |
| --- | --- |
| Full Extremes-DT field, regional `area` and `grid` options | `extremes-dt/full-field/extremes-dt-earthkit-example-domain.ipynb`, `extremes-dt/full-field-post-processing/extremes-dt-earthkit-serverside-interpolation.ipynb` |
| Feature extraction (timeseries, bounding box, polygon/country, vertical profile, trajectory, wave) | `extremes-dt/feature-extraction/*.ipynb` |
| On-Demand Extremes-DT (DEODE) output: `dataset: on-demand-extremes-dt`, `expver: 0099`, per-run `georef` | `on-demand-extremes-dt/full-field/` and `on-demand-extremes-dt/feature-extraction/` |
| DESP token (`~/.polytopeapirc`) | `desp-authentication.py` |

Operational limits from that README: 50 requests per second and at most 5 concurrent downloads per
user; data are served from LUMI (`polytope.lumi.apps.dte.destination-earth.eu`) and MN5
(`polytope.mn5.apps.dte.destination-earth.eu`). Relative dates such as `"date": "-14"` are accepted.
